In [ ]:
# HW12 – временные ряды: temporal split, baseline-модели и GRU-прогноз

In [ ]:
## 1. Импорты, seed и среда
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
## 2. Данные и первичный анализ
df = pd.read_csv('S12-hw-dataset.csv')
print(f"Размер датасета: {df.shape}")

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Пропуски:\n", df.isnull().sum())

print(f"Диапазон дат: с {df['date'].min()} по {df['date'].max()}")
print(f"Частота: {pd.infer_freq(df['date'])}")

plt.figure(figsize=(12,4))
plt.plot(df['date'], df['target'], linewidth=0.8)
plt.title('Временной ряд target')
plt.xlabel('Дата')
plt.ylabel('target')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('homeworks/HW12/artifacts/figures/initial_series.png', dpi=150)
plt.show()

print("Ряд имеет суточную сезонность (пик днём, спад ночью) и некоторые выбросы (например, резкие падения в выходные). Тренд слабовыражен.")

In [ ]:
## 3. Корректный temporal split
n = len(df)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train: {train_df['date'].min()} – {train_df['date'].max()}  ({len(train_df)} obs)")
print(f"Validation: {val_df['date'].min()} – {val_df['date'].max()}  ({len(val_df)} obs)")
print(f"Test: {test_df['date'].min()} – {test_df['date'].max()}  ({len(test_df)} obs)")

# Визуализация split
plt.figure(figsize=(12,5))
plt.plot(train_df['date'], train_df['target'], label='Train', linewidth=0.8)
plt.plot(val_df['date'], val_df['target'], label='Validation', linewidth=0.8)
plt.plot(test_df['date'], test_df['target'], label='Test', linewidth=0.8)
plt.axvline(x=train_df['date'].iloc[-1], color='gray', linestyle='--', alpha=0.7)
plt.axvline(x=val_df['date'].iloc[-1], color='gray', linestyle='--', alpha=0.7)
plt.title('Temporal split: train / validation / test')
plt.xlabel('Дата')
plt.ylabel('target')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('homeworks/HW12/artifacts/figures/series_split.png', dpi=150)
plt.show()

In [ ]:
## 4. Признаки для baseline-моделей
def create_features(df, lags=[1,7,14], rolling_windows=[7]):
    """
    Создаёт лаговые, rolling- и календарные признаки.
    Принимает DataFrame с колонками 'date' и 'target'.
    Возвращает DataFrame с признаками и целевой переменной (без пропусков).
    """
    df = df.copy()
    for lag in lags:
        df[f'lag_{lag}'] = df['target'].shift(lag)
    for win in rolling_windows:
        df[f'rolling_mean_{win}'] = df['target'].rolling(window=win, min_periods=1).mean()
        df[f'rolling_std_{win}'] = df['target'].rolling(window=win, min_periods=1).std()
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df = df.dropna().reset_index(drop=True)
    return df

train_feat = create_features(train_df)
val_feat = create_features(val_df)
test_feat = create_features(test_df)

print("Признаки созданы. Размеры:")
print(f"Train: {train_feat.shape}, Val: {val_feat.shape}, Test: {test_feat.shape}")

feature_cols = [c for c in train_feat.columns if c not in ['date', 'target']]
print("Используемые признаки:", feature_cols)

scaler = StandardScaler()
train_feat_scaled = scaler.fit_transform(train_feat[feature_cols])
val_feat_scaled = scaler.transform(val_feat[feature_cols])
test_feat_scaled = scaler.transform(test_feat[feature_cols])

X_train = train_feat_scaled
y_train = train_feat['target'].values
X_val = val_feat_scaled
y_val = val_feat['target'].values
X_test = test_feat_scaled
y_test = test_feat['target'].values

In [ ]:
## 5. Метрики и baseline модели
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
    return mae, rmse, mape

# B1: naive-last
last_train_value = train_feat['target'].iloc[-1]
y_pred_naive_val = np.full_like(y_val, last_train_value)
mae_b1_val, rmse_b1_val, mape_b1_val = calculate_metrics(y_val, y_pred_naive_val)
print(f"B1 (naive-last) val MAE: {mae_b1_val:.3f}, RMSE: {rmse_b1_val:.3f}, MAPE: {mape_b1_val:.2f}%")

# B2: moving-average
window = 7
last_7_mean = train_feat['target'].iloc[-window:].mean()
y_pred_ma_val = np.full_like(y_val, last_7_mean)
mae_b2_val, rmse_b2_val, mape_b2_val = calculate_metrics(y_val, y_pred_ma_val)
print(f"B2 (moving-average) val MAE: {mae_b2_val:.3f}, RMSE: {rmse_b2_val:.3f}, MAPE: {mape_b2_val:.2f}%")

# B3: Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge_val = ridge.predict(X_val)
mae_b3_val, rmse_b3_val, mape_b3_val = calculate_metrics(y_val, y_pred_ridge_val)
print(f"B3 (Ridge) val MAE: {mae_b3_val:.3f}, RMSE: {rmse_b3_val:.3f}, MAPE: {mape_b3_val:.2f}%")

In [ ]:
## 6. Оконное представление для GRU
class TimeSeriesDataset(Dataset):
    def __init__(self, data, window_size):
        self.data = torch.FloatTensor(data)
        self.window_size = window_size
        
    def __len__(self):
        return len(self.data) - self.window_size
    
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.window_size, :]
        y = self.data[idx+self.window_size, 0]
        return x, y

scaler_target = StandardScaler()
train_target_scaled = scaler_target.fit_transform(train_df['target'].values.reshape(-1,1)).flatten()
val_target_scaled = scaler_target.transform(val_df['target'].values.reshape(-1,1)).flatten()
test_target_scaled = scaler_target.transform(test_df['target'].values.reshape(-1,1)).flatten()

window_size = 24
batch_size = 64

train_dataset = TimeSeriesDataset(train_target_scaled.reshape(-1,1), window_size)
val_dataset = TimeSeriesDataset(val_target_scaled.reshape(-1,1), window_size)
test_dataset = TimeSeriesDataset(test_target_scaled.reshape(-1,1), window_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

In [ ]:
## 7. Модель GRU и обучение
class GRUForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

model = GRUForecaster(input_size=1, hidden_size=64, num_layers=2, dropout=0.2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate_model(model, loader, scaler):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y_pred = model(x).cpu().numpy()
            preds.extend(y_pred)
            trues.extend(y.numpy())
    preds = scaler.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
    trues = scaler.inverse_transform(np.array(trues).reshape(-1,1)).flatten()
    return calculate_metrics(trues, preds)

num_epochs = 50
patience = 10
best_val_mae = float('inf')
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'val_mae': []}

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    
    val_mae, val_rmse, val_mape = evaluate_model(model, val_loader, scaler_target)
    history['train_loss'].append(avg_loss)
    history['val_mae'].append(val_mae)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | Train Loss: {avg_loss:.4f} | Val MAE: {val_mae:.3f}")
    
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_epoch = epoch
        torch.save(model.state_dict(), 'homeworks/HW12/artifacts/best_gru.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping after {epoch+1} epochs")
            break

model.load_state_dict(torch.load('homeworks/HW12/artifacts/best_gru.pt'))
print(f"\nBest model from epoch {best_epoch+1} with Val MAE: {best_val_mae:.3f}")

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='Train Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1,2,2)
plt.plot(history['val_mae'], label='Validation MAE', color='orange')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Validation MAE')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('homeworks/HW12/artifacts/figures/gru_learning_curves.png', dpi=150)
plt.show()

test_mae, test_rmse, test_mape = evaluate_model(model, test_loader, scaler_target)
print(f"\nGRU Test MAE: {test_mae:.3f}, RMSE: {test_rmse:.3f}, MAPE: {test_mape:.2f}%")

In [ ]:
## 8. Сбор результатов в runs.csv
y_pred_naive_test = np.full_like(y_test, last_train_value)
mae_b1_test, rmse_b1_test, mape_b1_test = calculate_metrics(y_test, y_pred_naive_test)

y_pred_ma_test = np.full_like(y_test, last_7_mean)
mae_b2_test, rmse_b2_test, mape_b2_test = calculate_metrics(y_test, y_pred_ma_test)

y_pred_ridge_test = ridge.predict(X_test)
mae_b3_test, rmse_b3_test, mape_b3_test = calculate_metrics(y_test, y_pred_ridge_test)

results = pd.DataFrame([
    {
        'experiment_id': 'B1',
        'task': 'forecasting',
        'dataset': 'S12-hw-dataset.csv',
        'seed': SEED,
        'split_summary': f'train:{len(train_df)} val:{len(val_df)} test:{len(test_df)}',
        'window_size': None,
        'horizon': 1,
        'model_summary': 'naive-last',
        'features_summary': 'none',
        'scaler': 'none',
        'optimizer': None,
        'lr': None,
        'epochs_trained': None,
        'best_val_mae': mae_b1_val,
        'best_val_rmse': rmse_b1_val,
        'best_val_mape': mape_b1_val,
        'test_mae': mae_b1_test,
        'test_rmse': rmse_b1_test,
        'test_mape': mape_b1_test,
        'notes': ''
    },
    {
        'experiment_id': 'B2',
        'task': 'forecasting',
        'dataset': 'S12-hw-dataset.csv',
        'seed': SEED,
        'split_summary': f'train:{len(train_df)} val:{len(val_df)} test:{len(test_df)}',
        'window_size': 7,
        'horizon': 1,
        'model_summary': 'moving-average (window=7)',
        'features_summary': 'none',
        'scaler': 'none',
        'optimizer': None,
        'lr': None,
        'epochs_trained': None,
        'best_val_mae': mae_b2_val,
        'best_val_rmse': rmse_b2_val,
        'best_val_mape': mape_b2_val,
        'test_mae': mae_b2_test,
        'test_rmse': rmse_b2_test,
        'test_mape': mape_b2_test,
        'notes': ''
    },
    {
        'experiment_id': 'B3',
        'task': 'forecasting',
        'dataset': 'S12-hw-dataset.csv',
        'seed': SEED,
        'split_summary': f'train:{len(train_df)} val:{len(val_df)} test:{len(test_df)}',
        'window_size': None,
        'horizon': 1,
        'model_summary': 'Ridge(alpha=1)',
        'features_summary': f'{len(feature_cols)} features: {feature_cols}',
        'scaler': 'StandardScaler',
        'optimizer': None,
        'lr': None,
        'epochs_trained': None,
        'best_val_mae': mae_b3_val,
        'best_val_rmse': rmse_b3_val,
        'best_val_mape': mape_b3_val,
        'test_mae': mae_b3_test,
        'test_rmse': rmse_b3_test,
        'test_mape': mape_b3_test,
        'notes': ''
    },
    {
        'experiment_id': 'R1',
        'task': 'forecasting',
        'dataset': 'S12-hw-dataset.csv',
        'seed': SEED,
        'split_summary': f'train:{len(train_df)} val:{len(val_df)} test:{len(test_df)}',
        'window_size': window_size,
        'horizon': 1,
        'model_summary': 'GRU(hidden=64, layers=2, dropout=0.2)',
        'features_summary': 'univariate target',
        'scaler': 'StandardScaler (target)',
        'optimizer': 'Adam',
        'lr': 0.001,
        'epochs_trained': best_epoch+1,
        'best_val_mae': best_val_mae,
        'best_val_rmse': None,
        'best_val_mape': None,
        'test_mae': test_mae,
        'test_rmse': test_rmse,
        'test_mape': test_mape,
        'notes': ''
    }
])

os.makedirs('homeworks/HW12/artifacts', exist_ok=True)
results.to_csv('homeworks/HW12/artifacts/runs.csv', index=False)
print("runs.csv сохранён")

In [ ]:
## 9. Сохранение конфига лучшей GRU
config = {
    'model_type': 'GRU',
    'input_size': 1,
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.2,
    'window_size': window_size,
    'batch_size': batch_size,
    'learning_rate': 0.001,
    'optimizer': 'Adam',
    'loss': 'MSE',
    'seed': SEED,
    'scaler_target_mean': float(scaler_target.mean_[0]),
    'scaler_target_scale': float(scaler_target.scale_[0]),
    'best_val_mae': best_val_mae,
    'test_mae': test_mae
}
with open('homeworks/HW12/artifacts/best_gru_config.json', 'w') as f:
    json.dump(config, f, indent=4)
print("Конфиг сохранён")

In [ ]:
## 10. Финальный прогноз на test (лучшая модель)
model.eval()
all_preds, all_trues = [], []
for x, y in test_loader:
    x = x.to(device)
    pred = model(x).cpu().numpy()
    all_preds.extend(pred)
    all_trues.extend(y.numpy())
all_preds = scaler_target.inverse_transform(np.array(all_preds).reshape(-1,1)).flatten()
all_trues = scaler_target.inverse_transform(np.array(all_trues).reshape(-1,1)).flatten()

test_dates = test_df['date'].iloc[window_size:].reset_index(drop=True)

plt.figure(figsize=(12,5))
plt.plot(test_dates, all_trues, label='Actual', linewidth=1)
plt.plot(test_dates, all_preds, label='GRU Forecast', linewidth=1, alpha=0.8)
plt.title('Лучшая модель (GRU) – прогноз на test')
plt.xlabel('Дата')
plt.ylabel('target')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('homeworks/HW12/artifacts/figures/best_forecast_test.png', dpi=150)
plt.show()

In [ ]:
## 11. Сравнение baselines на validation
models = ['B1 (naive)', 'B2 (MA7)', 'B3 (Ridge)', 'R1 (GRU)']
val_maes = [mae_b1_val, mae_b2_val, mae_b3_val, best_val_mae]

plt.figure(figsize=(8,5))
bars = plt.bar(models, val_maes, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.ylabel('MAE')
plt.title('Сравнение моделей на валидации')
for bar, val in zip(bars, val_maes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.2f}', ha='center')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('homeworks/HW12/artifacts/figures/baselines_compare.png', dpi=150)
plt.show()

print("\nВсе артефакты сохранены в homeworks/HW12/artifacts/")